In [10]:
import pandas as pd
import numpy as np

In [3]:
url = "https://raw.githubusercontent.com/hankike/Premier_Prediction/refs/heads/main/data/cleaned_premier.csv"

df_premier = pd.read_csv(url)

In [11]:
df_premier['Date'] = pd.to_datetime(
    df_premier['Date'],
    dayfirst=True
)

df_premier = df_premier.sort_values('Date').reset_index(drop=True)

In [12]:
df_premier[['Date', 'HomeTeam', 'AwayTeam', 'FTR']].head()

,Date,HomeTeam,AwayTeam,FTR
0,2019-08-09,Liverpool,Norwich,H
1,2019-08-10,West Ham,Man City,A
2,2019-08-10,Bournemouth,Sheffield United,D
3,2019-08-10,Burnley,Southampton,H
4,2019-08-10,Crystal Palace,Everton,D


In [13]:
df_premier['FTR'].value_counts(normalize=True) * 100

FTR
H    43.421053
A    32.932331
D    23.646617
Name: proportion, dtype: float64

In [14]:
home = df_premier[[
    'Date', 'HomeTeam', 'FTR',
    'FTHG', 'FTAG', 'HS', 'HST', 'HC', 'HF'
]].copy()

home.columns = [
    'Date', 'Team', 'Result',
    'Goals_Scored', 'Goals_Conceded',
    'Shots', 'Shots_Target', 'Corners', 'Fouls'
]

home['Points'] = home['Result'].map({
    'H': 3,
    'D': 1,
    'A': 0
})

In [15]:
away = df_premier[[
    'Date', 'AwayTeam', 'FTR',
    'FTAG', 'FTHG', 'AS', 'AST', 'AC', 'AF'
]].copy()

away.columns = [
    'Date', 'Team', 'Result',
    'Goals_Scored', 'Goals_Conceded',
    'Shots', 'Shots_Target', 'Corners', 'Fouls'
]

away['Points'] = away['Result'].map({
    'A': 3,
    'D': 1,
    'H': 0
})

In [17]:
team_matches = pd.concat(
    [home, away],
    ignore_index=True
)

team_matches = team_matches.sort_values(
    ['Team', 'Date']
).reset_index(drop=True)

In [19]:
stats = [
    'Goals_Scored',
    'Goals_Conceded',
    'Shots',
    'Shots_Target',
    'Corners',
    'Fouls',
    'Points'
]

for stat in stats:
    team_matches[f'Avg_{stat}'] = (
        team_matches.groupby('Team')[stat]
        .transform(
            lambda x: x.shift(1).rolling(5, min_periods=1).mean()
        )
    )

In [21]:
df_premier['Season'] = (
    df_premier['Date'].dt.year.astype(str)
    + '-'
    + (df_premier['Date'].dt.year + 1).astype(str)
)

df_premier['Season'].value_counts().sort_index()

Season
2019-2020    199
2020-2021    336
2021-2022    408
2022-2023    361
2023-2024    412
2024-2025    372
2025-2026    378
2026-2027    194
Name: count, dtype: int64

In [22]:
feature_cols = [
    'Avg_Goals_Scored',
    'Avg_Goals_Conceded',
    'Avg_Shots',
    'Avg_Shots_Target',
    'Avg_Corners',
    'Avg_Fouls',
    'Avg_Points'
]

team_features = team_matches[
    ['Date', 'Team'] + feature_cols
].copy()

In [23]:
home_features = team_features.copy()

home_features = home_features.rename(
    columns={
        col: f'Home_{col}'
        for col in feature_cols
    }
)

home_features = home_features.rename(
    columns={'Team': 'HomeTeam'}
)

In [24]:
away_features = team_features.copy()

away_features = away_features.rename(
    columns={
        col: f'Away_{col}'
        for col in feature_cols
    }
)

away_features = away_features.rename(
    columns={'Team': 'AwayTeam'}
)

In [25]:
df_model = df_premier.merge(
    home_features,
    on=['Date', 'HomeTeam'],
    how='left'
)

df_model = df_model.merge(
    away_features,
    on=['Date', 'AwayTeam'],
    how='left'
)

In [26]:
df_model.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,Home_Avg_Corners,Home_Avg_Fouls,Home_Avg_Points,Away_Avg_Goals_Scored,Away_Avg_Goals_Conceded,Away_Avg_Shots,Away_Avg_Shots_Target,Away_Avg_Corners,Away_Avg_Fouls,Away_Avg_Points
0,E0,2019-08-09,20:00,Liverpool,Norwich,4,1,H,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,2019-08-10,12:30,West Ham,Man City,0,5,A,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,2019-08-10,15:00,Bournemouth,Sheffield United,1,1,D,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,2019-08-10,15:00,Burnley,Southampton,3,0,H,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,2019-08-10,15:00,Crystal Palace,Everton,0,0,D,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
for col in feature_cols:
    df_model[f'Diff_{col}'] = (
        df_model[f'Home_{col}']
        - df_model[f'Away_{col}']
    )